Pima Indians Diabetes (Kaggle)

Context
This dataset is originally from the National Institute of Diabetes and Digestive and Kidney Diseases. The objective of the dataset is to diagnostically predict whether or not a patient has diabetes, based on certain diagnostic measurements included in the dataset. Several constraints were placed on the selection of these instances from a larger database. In particular, all patients here are females at least 21 years old of Pima Indian heritage.

Content
The datasets consists of several medical predictor variables and one target variable, Outcome. Predictor variables includes the number of pregnancies the patient has had, their BMI, insulin level, age, and so on.

In [44]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import StandardScaler
from sklearn.impute import KNNImputer
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score


diabetes = pd.read_csv('./diabetes.csv')

print(diabetes.head())

   Pregnancies  Glucose  BloodPressure  SkinThickness  Insulin   BMI  \
0            6      148             72             35        0  33.6   
1            1       85             66             29        0  26.6   
2            8      183             64              0        0  23.3   
3            1       89             66             23       94  28.1   
4            0      137             40             35      168  43.1   

   DiabetesPedigreeFunction  Age  Outcome  
0                     0.627   50        1  
1                     0.351   31        0  
2                     0.672   32        1  
3                     0.167   21        0  
4                     2.288   33        1  


In [45]:
diabetes.info()

<class 'pandas.DataFrame'>
RangeIndex: 768 entries, 0 to 767
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Pregnancies               768 non-null    int64  
 1   Glucose                   768 non-null    int64  
 2   BloodPressure             768 non-null    int64  
 3   SkinThickness             768 non-null    int64  
 4   Insulin                   768 non-null    int64  
 5   BMI                       768 non-null    float64
 6   DiabetesPedigreeFunction  768 non-null    float64
 7   Age                       768 non-null    int64  
 8   Outcome                   768 non-null    int64  
dtypes: float64(2), int64(7)
memory usage: 54.1 KB


In [46]:
diabetes.describe()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
count,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000
mean,3.845052,120.894531,69.105469,20.536458,79.799479,31.992578,0.471876,33.240885,0.348958
std,3.369578,31.972618,19.355807,15.952218,115.244002,7.884160,0.331329,11.760232,0.476951
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.078000,21.000000,0.000000
25%,1.000000,99.000000,62.000000,0.000000,0.000000,27.300000,0.243750,24.000000,0.000000
50%,3.000000,117.000000,72.000000,23.000000,30.500000,32.000000,0.372500,29.000000,0.000000
75%,6.000000,140.250000,80.000000,32.000000,127.250000,36.600000,0.626250,41.000000,1.000000
max,17.000000,199.000000,122.000000,99.000000,846.000000,67.100000,2.420000,81.000000,1.000000


обработка пропусков (пропуски заполнены нулями в некоторых столбцах)

In [47]:
(diabetes==0).sum().sort_values(ascending=False)

Outcome                     500
Insulin                     374
SkinThickness               227
Pregnancies                 111
BloodPressure                35
BMI                          11
Glucose                       5
DiabetesPedigreeFunction      0
Age                           0
dtype: int64

In [48]:
cols = ["Insulin", "SkinThickness", "Glucose", "BloodPressure", "BMI"] #эти характеристики не могут быть нулями
#в столбцах Pregnancies и Outcome нули не являются пропусками
diabetes[cols] = diabetes[cols].replace(0, np.nan)
diabetes.isnull().sum().sort_values(ascending=False) #для использования Knn_imputer помечаем пропуски как nan

Insulin                     374
SkinThickness               227
BloodPressure                35
BMI                          11
Glucose                       5
Pregnancies                   0
DiabetesPedigreeFunction      0
Age                           0
Outcome                       0
dtype: int64

In [49]:
X_train, X_test, y_train, y_test = train_test_split(diabetes.drop(columns='Outcome'), diabetes['Outcome'], test_size=0.2, random_state=42)
sc = StandardScaler()

маштабируем признаки только по train, чтобы не было утечки данных

In [50]:
sc = StandardScaler()
X_train = sc.fit_transform(X_train)
X_test = sc.transform(X_test)


In [51]:
knn_imp = KNNImputer(n_neighbors=3)
X_train = knn_imp.fit_transform(X_train)
X_test = knn_imp.transform(X_test)

подбор лучшего параметра n_neighbors

In [52]:
scores = []
for i in range(1, 11):
    knn_model = KNeighborsClassifier(n_neighbors=i)
    knn_model.fit(X_train, y_train)
    y_pred = knn_model.predict(X_test)
    score = accuracy_score(y_test, y_pred)
    scores.append(score)

best_score = max(scores)
best_num = scores.index(best_score) + 1
print(best_score, best_num)

knn_model = KNeighborsClassifier(n_neighbors=best_num)
knn_model.fit(X_train, y_train)
y_pred = knn_model.predict(X_test)

print(accuracy_score(y_test, y_pred))


0.7662337662337663 9
0.7662337662337663


Кросс-валидация: пробуем разные комбинации разделения данных на train/test

In [53]:
from sklearn.model_selection import cross_val_score
results = cross_val_score(knn_model, X_test, y_test, cv=5)
print(results.mean())

0.7862365591397851
